# Lezione 17 - Creazione di Agenti AI Locali con Foundry Local e Qwen

In questo notebook costruirai un **assistente di ingegneria locale** che funziona interamente sulla tua workstation. Non viene utilizzata alcuna inferenza cloud in nessun momento. L'assistente può:

1. **Chiamare strumenti** tramite chiamate di funzione Qwen attraverso Foundry Local.
2. **Elencare e leggere file** all'interno di una directory di progetto isolata.
3. **Analizzare il codice** con metriche semplici.
4. **Cercare nella documentazione** con RAG locale (Chroma).
5. **Usare un server MCP locale** (saltato elegantemente se non configurato).

Il codice dell'agente appare quasi identico alle lezioni cloud — solo il client endpoint si sposta dal cloud a `localhost`.


## Configurazione

Prima di eseguire questo notebook:

1. **Installa Microsoft Foundry Local** (consulta la [documentazione](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) per il tuo sistema operativo).
2. **Scarica e avvia un modello Qwen:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Installa i pacchetti Python indicati di seguito.

Tutto viene eseguito localmente. Una macchina con circa 8 GB di RAM è un minimo realistico.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Connettersi a Foundry Local

`FoundryLocalManager` scarica il modello se necessario, avvia il servizio locale e ci fornisce un **endpoint compatibile con OpenAI**. Successivamente, puntiamo l'SDK standard di OpenAI su di esso. La chiave API è un segnaposto locale — non è coinvolta alcuna credenziale cloud.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Strumenti Locali (Operazioni su File in Sandbox)

Creiamo un piccolo progetto di esempio su disco, poi definiamo strumenti limitati a quella radice del progetto. Il controllo sandbox è importante anche localmente: uno strumento che legge percorsi arbitrari viene eseguito con le autorizzazioni del tuo utente e può accedere a tutto ciò che puoi.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. RAG Locale con Chroma

Inseriamo un piccolo insieme di frammenti di documentazione in una collezione Chroma locale. Chroma viene eseguito in-process e memorizza i vettori su disco — nessun server, nessun cloud. Lo strumento `search_docs` recupera i frammenti più rilevanti per una query.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. Il ciclo di chiamata degli strumenti

Ora registriamo gli strumenti con il modello utilizzando lo schema degli strumenti OpenAI ed eseguiamo il ciclo standard di chiamata degli strumenti — il modello richiede uno strumento, lo eseguiamo localmente, alimentiamo il risultato e ripetiamo finché il modello non produce una risposta finale. La funzione di chiamata affidabile di Qwen è ciò che permette che questo funzioni sul dispositivo.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. MCP Locale (Opzionale)

MCP è un trasporto, non un servizio cloud — un server MCP può essere eseguito come processo locale tramite `stdio`. La cella sottostante mostra come connettersi a un server MCP locale se ne hai uno configurato (ad esempio un server filesystem). Viene saltata con grazia quando `LOCAL_MCP_COMMAND` non è impostato, quindi il notebook funziona comunque dall'inizio alla fine senza di esso.

Nota di sicurezza: un server MCP locale viene eseguito con i permessi dell'utente. Limitane l'ambito a una directory di progetto e convalida i suoi output prima di agire su di essi.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Riepilogo

Hai costruito un assistente di ingegneria che funziona interamente sulla tua macchina:

- **Foundry Local** ha servito un modello **Qwen** dietro un endpoint compatibile con OpenAI — quindi il codice agente corrisponde alle lezioni cloud.
- **Strumenti in sandbox** hanno dato all'agente accesso ai file e analisi del codice senza uscire dalla cartella del progetto.
- **Chroma** ha fornito **RAG locale** sulla documentazione.
- **Local MCP** ha mostrato come riutilizzare l'ecosistema MCP offline.

Non è stato usato alcun inferenza cloud in alcun momento.

### Sfida

Estendi l'agente locale per:

1. **Lavorare con più server MCP** — connetti un server filesystem e un server git e lascia che l'agente scelga tra di essi.
2. **Usare memoria locale** — conserva una breve cronologia delle conversazioni su disco così l'assistente ricorda i turni precedenti tra riavvii del notebook.
3. **Supportare l'orchestrazione multi-agente locale** — aggiungi un secondo agente locale (ad esempio un revisore) e fa collaborare i due su un'attività.

Nella prossima lezione imparerai come mettere in sicurezza gli agenti AI dispiegati.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
